# 04 — Inference

Runs the treated-unit inferential battery from [methodology.md §5e](methodology.md):

1. **In-space placebo** — refit treating each donor as treated; compute Brent's permutation p-value
2. **In-time placebo** — refit with $T^{\text{fake}}_0$ = 6 months before real $T_0$
3. **Leave-one-donor-out** — drop each donor with weight > 0.05, recompute the gap, report stability

**Inputs**: requires `02_Fit_Models` to have run (so fits exist in `data/results/`).  
**Outputs**: `data/validation/inference_{test}_{event}_{model}.csv`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import T0, T0_FAKE, MODEL_HPARAMS, DONOR_POOL_VARIANT, LOO_MIN_WEIGHT
from lib.data import build_panel, load_fit, save_validation_table
from lib.validation import in_space_placebo, in_time_placebo, leave_one_out, gap_distribution

EVENTS = ['russia', 'hormuz']
WINDOW = 'preferred'
VARIANT = DONOR_POOL_VARIANT
MODELS = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']

print(f'Inferential battery: {EVENTS} × {MODELS}')

Inferential battery: ['russia', 'hormuz'] × ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']


## §5e (i) — In-space placebo

For each model: refit treating each donor as the placebo treated unit; compute the post/pre RMSPE ratio. Brent's rank in the placebo distribution gives the permutation p-value (< 0.10 is the Abadie convention).

*(Note: this is computationally heavy — N_donors × N_models × N_events fits. For convex SCM each placebo takes ~3s; for XGBoost/Bayesian Ridge much faster. Total runtime ~5-10 min.)*

In [2]:
import time
from lib.validation import get_tuned_hparams

iso_results = {}
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS:
        # Use the val-tuned hyperparameters from 02_Fit_Models so placebo runs match the headline fit.
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        if model in ('convex_scm', 'ascm'):
            kwargs['n_random_v'] = 40   # cut down for placebo speed
        t0 = time.time()
        try:
            df = in_space_placebo(model, panel, 'Brent', meta['donors'],
                                  t0=meta['t0'], t_pre_start=meta['t_pre_start'], **kwargs)
            iso_results[(event, model)] = df
            save_validation_table(df, f'inference_inspace_{event}_{model}')
            brent_p = float(df.loc['Brent', 'p_value']) if 'Brent' in df.index else np.nan
            print(f'  {event:6s} / {model:12s}  Brent p = {brent_p:.3f}  ({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'  {event:6s} / {model:12s}  ERROR: {str(e)[:60]}')

  russia / convex_scm    Brent p = 0.526  (73s)


  russia / ascm          Brent p = 0.579  (72s)


  russia / elastic_net   Brent p = 0.421  (0s)


  russia / xgboost       Brent p = 0.158  (5s)
  russia / bayesian_ridge  Brent p = 0.632  (0s)


  hormuz / convex_scm    Brent p = 0.053  (80s)


  hormuz / ascm          Brent p = 0.053  (80s)
  hormuz / elastic_net   Brent p = 0.053  (0s)


  hormuz / xgboost       Brent p = 0.053  (5s)
  hormuz / bayesian_ridge  Brent p = 0.053  (0s)


In [3]:
# Summary: Brent's p-value across models × events
rows = []
for (event, model), df in iso_results.items():
    if 'Brent' not in df.index:
        continue
    row = df.loc['Brent'].to_dict()
    row.update({'event': event, 'model': model})
    rows.append(row)

iso_brent = pd.DataFrame(rows)
if len(iso_brent):
    save_validation_table(iso_brent, 'inference_inspace_brent_summary')
    iso_brent.round(4)
else:
    print('No in-space placebo results to summarize.')

## §5e (ii) — In-time placebo

Refit with fake $T_0$ = 6 months before the real event. The gap in the fake post-period (from $T^{\text{fake}}_0$ to real $T_0$) should be small — large gap means the SCM is finding spurious effects.

In [4]:
from lib.validation import get_tuned_hparams

intime_rows = []
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    t0_fake = T0_FAKE[event]
    for model in MODELS:
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        try:
            r = in_time_placebo(model, panel, 'Brent', meta['donors'],
                                t0_fake=t0_fake, t_pre_start=meta['t_pre_start'], **kwargs)
            # Compute gap in the fake post-period (t0_fake to real t0)
            fake_post = r['gap'][(r['gap'].index >= t0_fake) & (r['gap'].index < meta['t0'])]
            mean_fake_gap_pct = float(100 * (np.exp(fake_post.mean()) - 1)) if len(fake_post) > 0 else np.nan
            intime_rows.append({
                'event': event, 'model': model,
                't0_fake': str(t0_fake.date()),
                'fake_post_obs': len(fake_post),
                'mean_fake_gap_pct': mean_fake_gap_pct,
                'pre_rmspe_log': r['rmspe_pre'],
            })
        except Exception as e:
            intime_rows.append({'event': event, 'model': model, 'error': str(e)[:60]})

intime_df = pd.DataFrame(intime_rows)
save_validation_table(intime_df, 'inference_intime')
intime_df.round(4)

,event,model,t0_fake,fake_post_obs,mean_fake_gap_pct,pre_rmspe_log
0,russia,convex_scm,2021-08-24,129,5.7452,0.1209
1,russia,ascm,2021-08-24,129,1.2669,0.1209
2,russia,elastic_net,2021-08-24,129,-0.9807,0.0543
3,russia,xgboost,2021-08-24,129,17.7535,0.0387
4,russia,bayesian_ridge,2021-08-24,129,-3.1665,0.0340
5,hormuz,convex_scm,2025-08-28,129,-10.2182,0.0591
6,hormuz,ascm,2025-08-28,129,-6.6833,0.0591
7,hormuz,elastic_net,2025-08-28,129,4.8566,0.0498
8,hormuz,xgboost,2025-08-28,129,-6.7778,0.0457
9,hormuz,bayesian_ridge,2025-08-28,129,-1.5865,0.0360


## §5e (iii) — Leave-one-donor-out

Drop each high-weight donor and recompute the gap. Stability (small range around the baseline gap) means no single donor is driving the headline.

In [5]:
from lib.validation import get_tuned_hparams

for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS:
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        if model in ('convex_scm', 'ascm'):
            kwargs['n_random_v'] = 40
        try:
            loo_results = leave_one_out(model, panel, 'Brent', meta['donors'],
                                        t0=meta['t0'], t_pre_start=meta['t_pre_start'],
                                        min_weight=LOO_MIN_WEIGHT, **kwargs)
            dist = gap_distribution(loo_results, t0=meta['t0'])
            save_validation_table(dist, f'inference_loo_{event}_{model}')
            print(f'\n{event} / {model} (baseline + {len(dist)-1} leave-outs):')
            print(dist.round(3).to_string())
        except Exception as e:
            print(f'{event} / {model}  ERROR: {str(e)[:80]}')


russia / convex_scm (baseline + 2 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        33.686          37.600       59.311        3.432
Coffee           59.555          60.564       96.784       29.468
Sugar            32.810          37.014       58.340        3.735



russia / ascm (baseline + 15 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         12.125          16.544       46.477      -23.797
Silver            10.574          14.676       47.805      -27.224
Platinum           0.935           4.121       54.257      -34.772
Gold              10.316          14.470       42.546      -21.353
Coffee            10.383          13.856       44.920      -25.084
Sugar             14.717          19.089       49.699      -19.944
LiveCattle        11.725          15.905       47.112      -24.199
SP500             11.066          15.291       45.984      -25.258
Nikkei            10.819          15.084       48.546      -25.785
JPY               17.265          21.849       46.646      -18.113
CHF               11.979          16.367       46.671      -24.323
CNY               12.454          16.855       46.294      -23.227
KRW               1


russia / xgboost (baseline + 5 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         31.873          29.281       96.175        2.295
Coffee            34.307          31.733       95.510        3.983
Sugar             34.357          32.384       91.849        5.993
LiveCattle        31.332          28.738       82.485        5.148
SP500             23.035          23.542       48.765       -5.078
TLT               32.593          30.121       93.109        4.385

russia / bayesian_ridge (baseline + 17 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline        -10.135          -4.763       39.563      -51.957
Silver           -12.893          -6.640       40.405      -55.804
Platinum         -23.722         -18.431       41.593      -63.444
Gold              -7.140       


hormuz / convex_scm (baseline + 3 leave-outs):
           mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                             
_baseline        58.517          59.406      101.475       14.345
Sugar            54.140          58.436       95.610        7.985
JPY              62.369          63.819      106.728       16.252
TLT              57.111          58.359       99.483       14.704



hormuz / ascm (baseline + 12 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         56.341          58.892       96.331       15.367
Gold              55.856          58.822       95.101       13.970
Coffee            61.658          63.446      104.899       19.546
Sugar             57.965          62.154       97.717       15.139
LiveCattle        57.117          59.396       97.921       14.884
Nikkei            56.448          58.465       95.933       15.561
JPY               56.856          59.494       98.011       15.531
CHF               56.444          59.078       96.819       15.531
CNY               56.133          58.845       96.071       15.312
INR               56.747          59.105       96.703       15.395
KRW               56.438          59.126       96.842       15.300
ZAR               56.431          59.060       96.445       15.402
TLT               5


hormuz / xgboost (baseline + 7 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         55.564          59.246       98.194       11.392
Gold              55.544          59.266       98.469       10.916
Coffee            61.643          65.527      106.271       15.276
LiveCattle        56.355          60.077       99.479       11.481
JPY               56.351          60.073       99.474       11.478
CHF               55.422          59.112       98.026       11.267
INR               55.346          59.018       97.910       11.246
MXN               55.693          59.370       98.348       11.499

hormuz / bayesian_ridge (baseline + 16 leave-outs):
            mean_gap_pct  median_gap_pct  max_gap_pct  min_gap_pct
name                                                              
_baseline         46.748          50.740       84.078       12.363
Silver            47.772       